In [3]:
import sys
import os
PROJECT_ROOT = os.getcwd()
import sklearn
from sklearn import preprocessing
sys.path.append(PROJECT_ROOT)
import numpy as np
from os.path import dirname
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter
import tempfile


def OS_CNN_feature_data_writer(dataset_name, train_ratio, ind, X_train_ori, y_train_ori, X_test, y_test):
    father_path = os.path.join(
    PROJECT_ROOT,
    'Splitted_datasets',
    dataset_name,
    str(train_ratio),
    str(ind)
    )
    if not os.path.exists(father_path):
        os.makedirs(father_path)

    dictionary = {
        'X_train': X_train_ori,
        'y_train': y_train_ori,
        'X_test': X_test,
        'y_test': y_test,
    }

    save_path = os.path.join(father_path, f"{dataset_name}.npy")
    np.save(save_path, dictionary)



def load_ts_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    data_start = False
    X_list = []
    y_list = []
    
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('@data'):
            data_start = True
            continue
        if data_start:
            parts = line.split(':')
            if len(parts) < 2:
                continue  # Skip invalid lines
            label = parts[-1]
            series_dims = parts[:-1]
            dims = []
            for dim_str in series_dims:
                try:
                    vals = [float(v.strip()) for v in dim_str.split(',') if v.strip()]
                    dims.append(vals)
                except ValueError:
                    print(f"Warning: Invalid value in {file_path}, skipping dimension.")
                    continue
            if dims:
                X_list.append(np.array(dims))  # Shape: (dimensions, length)
                y_list.append(label)
    
    return X_list, y_list


def data_loader(dataset_name):
    # Define base path for dataset
    base_path = os.path.join(
    PROJECT_ROOT,
    'datasets',
    'UCRArchive_2018',
    dataset_name
    )
    # Check for .ts file extension first
    train_file = os.path.join(base_path, f"{dataset_name}_TRAIN.ts")
    test_file = os.path.join(base_path, f"{dataset_name}_TEST.ts")
    
    if os.path.exists(train_file) and os.path.exists(test_file):
        print(f"Loading .ts train file: {train_file}")
        print(f"Loading .ts test file: {test_file}")
        
        # Load and parse .ts files
        X_train_list, y_train_str = load_ts_file(train_file)
        X_test_list, y_test_str = load_ts_file(test_file)
        
        # Encode labels
        le = preprocessing.LabelEncoder()
        le.fit(y_train_str)
        y_train = le.transform(y_train_str)
        y_test = le.transform(y_test_str)
        
        # Convert to np.float32
        X_train = [x.astype(np.float32) for x in X_train_list]
        X_test = [x.astype(np.float32) for x in X_test_list]
        
        return X_train, y_train, X_test, y_test
    
    # Fallback to .txt or .tsv
    train_file = os.path.join(base_path, f"{dataset_name}_TRAIN.txt")
    test_file = os.path.join(base_path, f"{dataset_name}_TEST.txt")
    delimiter = ' '  # Single space delimiter for .txt
    
    if not os.path.exists(train_file):
        train_file = os.path.join(base_path, f"{dataset_name}_TRAIN.tsv")
        test_file = os.path.join(base_path, f"{dataset_name}_TEST.tsv")
        delimiter = '\t'  # Tab delimiter for .tsv
    
    # Print file paths for debugging
    print(f"Loading train file: {train_file}")
    print(f"Loading test file: {test_file}")
    
    # Function to clean file content
    def clean_file(input_path, output_path):
        with open(input_path, 'r') as f_in, open(output_path, 'w') as f_out:
            for i, line in enumerate(f_in):
                # Split line by delimiter, remove empty or invalid tokens
                tokens = [token.strip() for token in line.strip().split(delimiter) if token.strip()]
                if not tokens:  # Skip empty lines
                    print(f"Warning: Empty line found at row {i} in {input_path}, skipping.")
                    continue
                # Replace empty or non-numeric tokens with '0.0'
                cleaned_tokens = []
                for token in tokens:
                    try:
                        float(token)  # Check if token is numeric
                        cleaned_tokens.append(token)
                    except ValueError:
                        print(f"Warning: Non-numeric value '{token}' at row {i} in {input_path}, replacing with 0.0")
                        cleaned_tokens.append('0.0')
                # Ensure consistent number of columns (based on first valid row)
                if i == 0:
                    expected_cols = len(cleaned_tokens)
                if len(cleaned_tokens) != expected_cols:
                    print(f"Warning: Row {i} in {input_path} has {len(cleaned_tokens)} columns, expected {expected_cols}, padding/truncating.")
                    cleaned_tokens = cleaned_tokens[:expected_cols] + ['0.0'] * (expected_cols - len(cleaned_tokens))
                # Write cleaned line
                f_out.write(' '.join(cleaned_tokens) + '\n')
        return output_path
    
    # Create temporary cleaned files
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as temp_train, \
         tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as temp_test:
        cleaned_train_file = clean_file(train_file, temp_train.name)
        cleaned_test_file = clean_file(test_file, temp_test.name)
    
    # Inspect first few lines of cleaned files
    def inspect_file(file_path, num_lines=5):
        print(f"Inspecting first {num_lines} lines of cleaned {file_path}:")
        with open(file_path, 'r') as f:
            for i, line in enumerate(f):
                if i >= num_lines:
                    break
                print(f"Line {i}: {line.strip()}")
    
    inspect_file(cleaned_train_file)
    inspect_file(cleaned_test_file)
    
    # Load cleaned dataset
    try:
        Train_dataset = np.loadtxt(cleaned_train_file, delimiter=' ', dtype=np.float64, comments='#')
        print(f"Successfully loaded cleaned {cleaned_train_file}")
    except ValueError as e:
        print(f"Error loading cleaned train file {cleaned_train_file}: {e}")
        raise
    try:
        Test_dataset = np.loadtxt(cleaned_test_file, delimiter=' ', dtype=np.float64, comments='#')
        print(f"Successfully loaded cleaned {cleaned_test_file}")
    except ValueError as e:
        print(f"Error loading cleaned test file {cleaned_test_file}: {e}")
        raise
    
    # Clean up temporary files
    os.remove(cleaned_train_file)
    os.remove(cleaned_test_file)
    
    # Convert to float32
    Train_dataset = Train_dataset.astype(np.float32)
    Test_dataset = Test_dataset.astype(np.float32)
    
    # Split features and labels
    X_train = Train_dataset[:, 1:]
    y_train = Train_dataset[:, 0:1]
    
    X_test = Test_dataset[:, 1:]
    y_test = Test_dataset[:, 0:1]
    
    # Encode labels
    le = preprocessing.LabelEncoder()
    le.fit(np.squeeze(y_train, axis=1))
    y_train = le.transform(np.squeeze(y_train, axis=1))
    y_test = le.transform(np.squeeze(y_test, axis=1))
    
    # # Handle NaN values
    # X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    # X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
    
    return X_train, y_train, X_test, y_test



name_list = [
 'ECG200',
]


train_ratio_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]


    
for name in  name_list:
    X_train_ori, y_train_ori, X_test, y_test = data_loader(name)

    train_dist = Counter(y_train_ori)
    test_dist = Counter(y_test)
    # Print feature shapes
    print(f"Dataset: {name}")
    if isinstance(X_train_ori, list):
        print("x_train: list of", len(X_train_ori), "series")
        print("x_test: list of", len(X_test), "series")
    else:
        print("x_train:", X_train_ori.shape)
        print("x_test:", X_test.shape)
    print("y_train:", y_train_ori.shape)
    print("y_test:", y_test.shape)
    print(f"train classes: {len(train_dist)}")
    print(f"train: {train_dist}")
    print(f"test:  {test_dist}")
            
    OS_CNN_feature_data_writer(name, 1, 10, X_train_ori, y_train_ori, X_test, y_test)  
    
    for train_ratio in train_ratio_list:  
        print(f" Trying train_ratio = {train_ratio}...")  
        X_train_ori, y_train_ori, X_test, y_test = data_loader(name)
        count = Counter(y_train_ori)
        min_count = min(count.values())
        n_required = int(np.ceil(2 / train_ratio)) 

        if min_count < n_required:
            print(f"Skipping ratio {train_ratio} - some classes have < {n_required} samples.")
            continue
        sss = StratifiedShuffleSplit(n_splits=10, test_size=1 - train_ratio, random_state=0)
        sss.get_n_splits(X_train_ori, y_train_ori)
        ind = 0
        if isinstance(X_train_ori, list):
            dummy_X = np.zeros(len(X_train_ori))
            for train_index, test_index in sss.split(dummy_X, y_train_ori):
                print(ind)
                X_train = [X_train_ori[i] for i in train_index]
                y_train = y_train_ori[train_index]

                if isinstance(X_train, list):
                    print("x_train: list of", len(X_train), "Time series")
                    print("x_test: list of", len(X_test), "Time series")
                else:
                    print("x_train:", X_train.shape)
                    print("x_test:", X_test.shape)
                print("y_train:", y_train.shape)
                print("y_test:", y_test.shape)
                OS_CNN_feature_data_writer(name, train_ratio, ind, X_train, y_train, X_test, y_test)
                ind = ind+1
        else:
            for train_index, test_index in sss.split(X_train_ori, y_train_ori):
                print(ind)
                X_train = X_train_ori[train_index,:]
                y_train = y_train_ori[train_index]

                print("x_train:", X_train.shape)
                print("x_test:", X_test.shape)
                print("y_train:", y_train.shape)
                print("y_test:", y_test.shape)
                OS_CNN_feature_data_writer(name, train_ratio, ind, X_train, y_train, X_test, y_test)
                ind = ind+1

Loading train file: c:\Users\ADMIN\Desktop\osse-lstm\datasets\UCRArchive_2018\ECG200\ECG200_TRAIN.tsv
Loading test file: c:\Users\ADMIN\Desktop\osse-lstm\datasets\UCRArchive_2018\ECG200\ECG200_TEST.tsv
Inspecting first 5 lines of cleaned C:\Users\ADMIN\AppData\Local\Temp\tmpx895ll9k.txt:
Line 0: -1 0.50205548 0.54216265 0.72238348 1.4288852 2.1365158 2.281149 1.9362737 1.46889 1.0088451 0.38028224 -0.29677967 -0.51392868 -0.25564469 -0.10720254 -0.28782655 -0.41800901 -0.31916313 -0.2603787 -0.35035721 -0.50548599 -0.71088709 -0.82391982 -0.89970154 -1.1539497 -1.2298306 -1.044091 -1.2020312 -1.3921949 -1.1301083 -1.1798666 -1.6492718 -1.7265754 -1.6083704 -1.6628022 -1.6506724 -1.6973094 -1.8386968 -1.8025962 -1.7805361 -1.8251665 -1.6447633 -1.4238097 -1.3921949 -1.3604156 -1.2001781 -0.91863234 -0.68591581 -0.66794346 -0.51272154 -0.10169069 0.063954259 0.082614311 0.23760718 0.17479318 0.12320539 0.5033942 0.6838702 0.47499476 0.53279711 0.72354995 0.6644198 0.64793559 0.75705403 0